# Hazelnut Price Data — EDA

## Data sources

| Source | What it is | Coverage | Frequency |
|--------|-----------|----------|-----------|
| **TOBB** | Turkish commodity exchange (borsa) transaction records. Regulated exchanges where traders must register every physical trade. | 2005–present | Daily (trading days) |
| **TB.org** | Free-market (serbest piyasa) indicative quotes for three hazelnut varieties. Not a matched transaction — closer to a price survey. | 2019–2024 | Daily |

## What "transaction count" means

`transaction_count` is the **number of individual trade lots** recorded at an exchange on a given day — not the number of parties. Two farmers each selling a single lot = count of 2. Days with count=1 or count=2 are thin-market days: one large buyer vs. one seller. These days still produce a valid price, but the min/max spread can be wide (one negotiation, not a market).

High-count days (50+) reflect active harvest-season trading with many participants.

## Varieties

- **Tombul (Levant/Giresun)** — round, high oil content, the premium grade. This is what Ferrero buys. TOBB tracks it as *FINDIK KABUKLU TOMBUL (LEVANT)*.
- **Yağlı** — higher fat content, used in confectionery/chocolate
- **Sivri** — elongated, lower grade, lower price

TB.org reports all three. TOBB in this dataset is Tombul/Levant only (ana_kod=9, alt_kod=904).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.dpi': 130, 'font.size': 9})

combined = pd.read_csv('data/processed/hazelnut_combined.csv', parse_dates=['date'])
tobb     = combined[combined['source'] == 'tobb'].copy()
tb       = combined[combined['source'] == 'tb_org'].copy()

print(f"TOBB   : {len(tobb):,} rows | {tobb['exchange'].nunique()} exchanges | "
      f"{tobb['date'].min().date()} – {tobb['date'].max().date()}")
print(f"TB.org : {len(tb):,} rows | varieties: {sorted(tb['variety'].unique())} | "
      f"{tb['date'].min().date()} – {tb['date'].max().date()}")

## 1 · TOBB daily prices by exchange (2005–present)

Each dot is one trading day at one exchange. Price is `avg_price_tlkg` (TL/kg, in-shell Tombul). Note the two phases: pre-2021 low nominal prices (TL was stronger), post-2021 surge driven by TL depreciation + tight supply.

In [ ]:
exchanges  = sorted(tobb['exchange'].unique())
colors     = plt.cm.tab10(np.linspace(0, 0.6, len(exchanges)))
ex_color   = dict(zip(exchanges, colors))

fig, ax = plt.subplots(figsize=(12, 4))
for ex in exchanges:
    sub = tobb[tobb['exchange'] == ex].sort_values('date')
    ax.scatter(sub['date'], sub['avg_price_tlkg'],
               s=2, alpha=0.6, label=ex, color=ex_color[ex])

ax.set_ylabel('TL / kg (in-shell Tombul)')
ax.set_title('TOBB daily avg price by exchange')
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.legend(markerscale=4, framealpha=0.7, ncol=2, fontsize=8)
plt.tight_layout()
plt.show()

## 2 · TOBB monthly VWAP + trading activity

Volume-weighted average price (VWAP) across all exchanges. Bottom panel shows total transaction count per month — use this to judge how thin the market is in off-season months (Feb–Jul).

In [ ]:
tobb['ym'] = tobb['date'].dt.to_period('M')

def monthly_vwap(g):
    mask = g['volume_kg'].notna() & (g['volume_kg'] > 0)
    if mask.sum() > 0:
        return np.average(g.loc[mask, 'avg_price_tlkg'], weights=g.loc[mask, 'volume_kg'])
    return g['avg_price_tlkg'].mean()

monthly = (
    tobb.groupby('ym')
    .apply(lambda g: pd.Series({
        'vwap':   monthly_vwap(g),
        'txn_ct': g['transaction_count'].sum(),
        'n_days': g['date'].nunique(),
    }))
    .reset_index()
)
monthly['date'] = monthly['ym'].dt.to_timestamp()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 5), sharex=True,
                                gridspec_kw={'height_ratios': [3, 1]})

ax1.plot(monthly['date'], monthly['vwap'], lw=1.2, color='steelblue')
ax1.set_ylabel('TL / kg (VWAP)')
ax1.set_title('TOBB monthly VWAP — all exchanges combined')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

ax2.bar(monthly['date'], monthly['txn_ct'], width=20, color='steelblue', alpha=0.6)
ax2.set_ylabel('Total txn\ncount')

for ax in (ax1, ax2):
    ax.xaxis.set_major_locator(mdates.YearLocator(2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.show()

## 3 · TB.org variety prices

Free-market indicative quotes for Yağlı, Levant, and Sivri. Shaded band = min–max range on each date. Levant commands a consistent premium.

In [ ]:
variety_colors = {'Yağlı': '#e07b39', 'Levant': '#3a7ebf', 'Sivri': '#5aaa5a'}

fig, ax = plt.subplots(figsize=(11, 4))
for var, grp in tb.groupby('variety'):
    grp = grp.sort_values('date')
    c = variety_colors.get(var, 'gray')
    ax.plot(grp['date'], grp['avg_price_tlkg'], lw=1.4, label=var, color=c)
    ax.fill_between(grp['date'], grp['min_price_tlkg'], grp['max_price_tlkg'],
                    alpha=0.15, color=c)

ax.set_ylabel('TL / kg')
ax.set_title('TB.org free-market variety prices (shaded = min–max range)')
ax.legend()
ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 7]))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

variety_stats = (
    tb.groupby('variety')['avg_price_tlkg']
    .agg(['mean', 'median', 'std', 'count'])
    .rename(columns={'mean': 'mean_TLkg', 'median': 'median_TLkg',
                     'std': 'std_TLkg', 'count': 'n_days'})
    .round(1)
)
print(variety_stats)

## 4 · TOBB vs TB.org overlap (2019–2024)

Do the two sources agree? TOBB Giresun (primary Tombul exchange) vs TB.org Levant, as monthly averages. If they track together the sources are consistent.

In [ ]:
giresun = (
    tobb[tobb['exchange'].str.contains('Giresun', case=False, na=False)]
    .assign(ym=lambda d: d['date'].dt.to_period('M'))
    .groupby('ym')['avg_price_tlkg'].mean()
    .reset_index()
)
giresun['date'] = giresun['ym'].dt.to_timestamp()

tb_levant = (
    tb[tb['variety'] == 'Levant']
    .assign(ym=lambda d: d['date'].dt.to_period('M'))
    .groupby('ym')['avg_price_tlkg'].mean()
    .reset_index()
)
tb_levant['date'] = tb_levant['ym'].dt.to_timestamp()

fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(giresun['date'], giresun['avg_price_tlkg'],
        lw=1.5, label='TOBB Giresun (monthly avg)', color='steelblue')
ax.plot(tb_levant['date'], tb_levant['avg_price_tlkg'],
        lw=1.5, label='TB.org Levant (monthly avg)', color='#e07b39', ls='--')

ax.set_ylabel('TL / kg')
ax.set_title('Price continuity check: TOBB Giresun vs TB.org Levant')
ax.legend()
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.show()

## 5 · Continuity assessment — TOBB

Three diagnostics:
1. **Trading days per exchange per year** — heatmap. White = no data; look for the 2021–22 gap.
2. **Transaction count distribution** — how often is the market thin (≤2 transactions)?
3. **Annual summary table** — spread, volume, thin-day count.

In [ ]:
tobb['year'] = tobb['date'].dt.year
heatmap_data = (
    tobb.groupby(['exchange', 'year'])['date']
    .nunique()
    .unstack(fill_value=0)
)

fig, ax = plt.subplots(figsize=(14, 3.5))
im = ax.imshow(heatmap_data.values, aspect='auto', cmap='YlOrRd', vmin=0)
ax.set_xticks(range(len(heatmap_data.columns)))
ax.set_xticklabels(heatmap_data.columns, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(len(heatmap_data.index)))
ax.set_yticklabels(heatmap_data.index)
ax.set_title('Trading days per exchange per year (white = no data)')
plt.colorbar(im, ax=ax, label='days')
plt.tight_layout()
plt.show()

In [ ]:
tc    = tobb['transaction_count'].dropna()
thin  = (tc <= 2).sum()
total = len(tc)

fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(tc.clip(upper=50), bins=40, color='steelblue', edgecolor='white', lw=0.3)
ax.axvline(2, color='red', ls='--', lw=1, label=f'≤2 txn: {thin/total:.0%} of days')
ax.set_xlabel('Transaction count per day (clipped at 50)')
ax.set_ylabel('# trading days')
ax.set_title('TOBB: distribution of daily transaction counts')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Thin-market days (≤2 transactions): {thin:,} / {total:,}  ({thin/total:.1%})')

In [ ]:
tobb['spread_pct'] = (tobb['max_price_tlkg'] - tobb['min_price_tlkg']) / tobb['avg_price_tlkg'] * 100

annual = (
    tobb.groupby('year')
    .agg(
        trading_days  =('date',             'nunique'),
        avg_price     =('avg_price_tlkg',    'mean'),
        median_spread =('spread_pct',         'median'),
        total_vol_mt  =('volume_kg',          'sum'),
        thin_days     =('transaction_count',  lambda x: (x <= 2).sum()),
    )
    .assign(total_vol_mt=lambda d: d['total_vol_mt'] / 1000)
    .round({'avg_price': 1, 'median_spread': 1, 'total_vol_mt': 0})
)
annual.columns = ['trading_days', 'avg_TLkg', 'median_spread_%', 'total_vol_MT', 'thin_days']
print(annual.to_string())